# Projeto Extensionista - Dashboard de Enchentes em Porto Alegre - INCT SiMAI - FASE 2
**Nomes:** Amanda Flores Domeneghini, Milena Fontana Bregalda, Mariana Nunes dos Santos, Karen Cristiane Nascimento


**Disciplina**: Coleta, Preparação e Análise de Dados


**Turma**: 310/110

**Professora**: Katherine Bianchini Esper


**Data**: 09/11/2026

Dependências necessárias:

In [1]:
!pip install requests
!pip install pandas

Script:

Script pega o arquivo JSON com as coordenadas já convertidas das casas de bomba e faz um cruzamento de dados (join espacial) com a API de Cotas de Inundação da Prefeitura de Porto Alegre. Ele não pega os dados da API 2 completa porque eram muitos dados, chegava num JSON de 200MB e crashava. Assim pega apenas o que e necessario.

In [3]:

import requests
import json
import time

# Link da segunda API (Cotas de Inundacao):
url = "https://gis-smamus.portoalegre.rs.gov.br/server/rest/services/01_PUBLICACOES/cotas_inundacao_300_535/FeatureServer/0/query"

# Abre o JSON ja coletado e convertido da primeira API (Casas de Bomba - EBAP):
with open("../dados/processados/api_casasbombaebap_convertido.json", "r", encoding="utf-8") as f:
    dados_casas = json.load(f)

# Lista para guardar os resultados das consultas
resultados = []

# Percorre todas as casas de bomba do JSON
for casa in dados_casas["features"]:

    # Coleta o identificador e o nome da casa
    fid = casa["attributes"]["fid"]
    nome = casa["attributes"]["name"]

    # Pega as coordenadas originais da casa de bomba
    x = casa["geometry"]["x"]
    y = casa["geometry"]["y"]

    # Parametros de consulta da segunda API:
    parametros = {
        "geometry": f"{x},{y}",
        "geometryType": "esriGeometryPoint",
        "inSR": "4326",
        "spatialRel": "esriSpatialRelIntersects",
        "outFields": "objectid,contourmax",
        "returnGeometry": "false",
        "f": "json"
    }

    # Verifica se as coordenadas estao em longitude/latitude
    if not (-180 <= x <= 180 and -90 <= y <= 90):
        print("Erro: as coordenadas de", nome, "nao estao em longitude/latitude.")
        print("Eh necessario converter as coordenadas antes da consulta.")
        break

    try:
        resposta = requests.get(url, params=parametros, timeout=60)

        print("Status da requisicao para", nome, ":", resposta.status_code)

        # Tratamento de erros
        if resposta.status_code == 200:

            dados = resposta.json()

            if "error" in dados:
                print("Erro da API:", dados["error"])
                break

            if dados.get("exceededTransferLimit"):
                print("Erro: consulta incompleta para", nome)
                break

            # Guarda apenas os campos necessarios
            poligonos = []

            for poligono in dados["features"]:

                poligonos.append({
                    "objectid": poligono["attributes"]["objectid"],
                    "contourmax": poligono["attributes"]["contourmax"]
                })

            # Coleta todas as cotas encontradas
            cotas = [
                p["contourmax"]
                for p in poligonos
                if p["contourmax"] is not None
            ]

            # Identifica a menor cota encontrada
            menor_cota = min(cotas) if cotas else None

            # Salva os dados desta casa de bomba
            resultados.append({
                "fid": fid,
                "nome": nome,
                "longitude": x,
                "latitude": y,
                "poligonos": poligonos,
                "menor_cota_cm": menor_cota
            })

            print("Poligonos encontrados:", len(poligonos))
            print("Menor cota:", menor_cota)
            print("-----------------------------")

        else:
            print("Erro na requisicao:", resposta.text)
            break

    except requests.exceptions.RequestException as erro:
        print("Erro de conexao:", erro)
        break

    # Aguarda 2 segundos antes de consultar a proxima casa
    time.sleep(2)

# Salva o JSON somente se todas as casas foram consultadas
if len(resultados) == len(dados_casas["features"]):

    with open("../dados/originais/dadosapi_cotasinundacao.json", "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=4)

    print("Arquivo JSON salvo com sucesso!")
    print("Quantidade de casas analisadas:", len(resultados))

else:
    print("A coleta nao foi concluida. O arquivo nao foi salvo.")


Status da requisicao para CB-2 : 200
Poligonos encontrados: 12
Menor cota: 430.0
-----------------------------
Status da requisicao para CB-3 : 200
Poligonos encontrados: 19
Menor cota: 350.0
-----------------------------
Status da requisicao para CB-4 : 200
Poligonos encontrados: 24
Menor cota: 310.0
-----------------------------
Status da requisicao para CB-5 : 200
Poligonos encontrados: 18
Menor cota: 340.0
-----------------------------
Status da requisicao para CB-6 : 200
Poligonos encontrados: 12
Menor cota: 430.0
-----------------------------
Status da requisicao para CB-9 : 200
Poligonos encontrados: 25
Menor cota: 300.0
-----------------------------
Status da requisicao para CB-11B : 200
Poligonos encontrados: 7
Menor cota: 480.0
-----------------------------
Status da requisicao para CB-12 : 200
Poligonos encontrados: 24
Menor cota: 310.0
-----------------------------
Status da requisicao para CB-13 : 200
Poligonos encontrados: 25
Menor cota: 300.0
----------------------------